In [58]:
import random
import numpy as np

In [88]:
def read_data(filename):
  data = np.ndarray([12, 22, 3])

  with open(filename, 'r') as f:
    i = 0
    for row in f:
      i += 1
      if i == 1:
        continue

      items = row.split(",")
      hits = int(items[3])
      stands = int(items[4])

      if hits == 0 and stands == 0:
        # never seen in actual gameplay
        data[int(items[0])][int(items[1])][int(items[2])] = 0.5
      else:
        data[int(items[0])][int(items[1])][int(items[2])] = hits / (hits + stands)

  return data

In [59]:
def random_card():
  cards = [2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10, 11]
  return random.choice(cards)

In [132]:
def bot_choice(args, value, num_aces):
  player_choices = args[0]
  dealer_card = args[1]

  if value == 21:
    return 0

  if random.random() < player_choices[dealer_card][value][num_aces]:
    return 1
  else:
    return 0

def dealer_choice(args, value, num_aces):
  if value < 17:
    return 1
  else:
    return 0

def play_round(card1, card2, choice_algorithm, deck, other_args):
  choice_list = []
  card_list = [card1, card2]
  num_aces = 0
  aces_used = 0
  if card1 == 11:
    num_aces += 1
  if card2 == 11:
    num_aces += 1
  value = card1 + card2
  if value > 21:
    value -= 10
    aces_used += 1

  while True:
    choice = choice_algorithm(other_args, value, min(num_aces, 2))
    choice_list.append((value, min(num_aces, 2), choice, card_list.copy()))
    if choice == 0:
      # stand
      break

    added_card = deck.pop(0)
    card_list.append(added_card)
    if added_card == 11:
      num_aces += 1

    value += added_card
    while value > 21 and num_aces - aces_used > 0:
      value -= 10
      aces_used += 1

    if value > 21:
      # bust
      choice_list.append((value, min(num_aces, 2), 2, card_list))
      break

  return choice_list, value

In [153]:
def play_blackjack(agent, player_choices, num_other_players):
  static_deck = []
  for i in range(0, 52):
    static_deck.append(random_card())

  num_wins = 0
  num_ties = 0
  num_losses = 0
  for i in range(0, 1000000):
    deck = static_deck.copy()
    random.shuffle(deck)
    dealer_card = deck.pop(0)
    choice_list = []
    end_of_round_values = []
    for i in range(0, num_other_players):
      card1 = deck.pop(0)
      card2 = deck.pop(0)
      choices, value = play_round(card1, card2, bot_choice, deck, (player_choices, dealer_card))
      choice_list.extend(choices)
      end_of_round_values.append(value)

    # FEED choice_list AND player_choices INTO agent() HERE
    # (before dealer gets to play)

    card2 = deck.pop()
    _, value = play_round(dealer_card, card2, dealer_choice, deck, None)

    for player in end_of_round_values:
      if player > 21:
        if value > 21:
          num_ties += 1
        else:
          num_losses += 1
      elif value > 21:
        num_wins += 1
      elif player > value:
        num_wins += 1
      elif player == value:
        num_ties += 1
      else:
        num_losses += 1

  print(num_wins, num_ties, num_losses)

In [120]:
def random_play():
  return random.randint(0, 1)

In [154]:
data = read_data("blkjck_clean.csv")
play_blackjack(random_play, data, 5)

2070283 631502 2298215
